In [ ]:
from pathlib import Path
import re
import yaml
import os
from openpyxl import load_workbook

In [ ]:
!date
jupyter_dir = os.getcwd()
print(f"DIR: {jupyter_dir}")

In [ ]:
#!date
#excel_file = Path("./Subscriber.xlsx")
#out_yaml = Path("./output2/output.yaml")

In [ ]:
!date
from pathlib import Path
import re
import yaml
from openpyxl import load_workbook

# ====== Excel layout ======
SHEETS = ["SourceAddress", "BlackList", "WhiteList", "CategoryList"]

ROW_IDENTIFIER = 4
ROW_DATA_START = 5
COL_START = 3

# ====== BIG-IP fixed rules ======
MASK_FIXED = "255.255.255.255"
VS_RULE = "/Common/Shared/url_filter_irule"
TMC_DESTINATION_ADDRESS_INLINE = "192.168.23.12"

TCP_PROFILES = [
    "/Common/Shared/{id}_Profile",
    "/Common/Shared/{id}_stats",
    "/Common/tcp",
]

UDP_PROFILES = [
    "/Common/Shared/{id}_Profile",
    "/Common/Shared/{id}_stats",
    "/Common/Shared/udp_gtm_dns",
]

split_pattern = re.compile(r"[,\n\r]+")


def split_values(v):
    if v is None:
        return []
    s = str(v).strip()
    if not s:
        return []
    return [x.strip() for x in split_pattern.split(s) if x.strip()]


def read_identifier_columns(ws):
    col_to_id = {}
    for col in range(COL_START, ws.max_column + 1):
        v = ws.cell(row=ROW_IDENTIFIER, column=col).value
        if v is None:
            continue
        ident = str(v).strip()
        if ident:
            col_to_id[col] = ident
    return col_to_id


def read_sheet_values_by_identifier(ws):
    col_to_id = read_identifier_columns(ws)
    result = {ident: [] for ident in col_to_id.values()}

    for col, ident in col_to_id.items():
        values = []
        for row in range(ROW_DATA_START, ws.max_row + 1):
            cell_v = ws.cell(row=row, column=col).value
            values.extend(split_values(cell_v))

        values = [v for v in values if v and v.strip()]
        result[ident] = sorted(set(values))

    return result


def build_virtual_servers_expected(identifier: str):
    return {
        "TCP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_TCP",
            "ip_protocol": "tcp",
            "mask": MASK_FIXED,
            "profiles": sorted({p.format(id=identifier) for p in TCP_PROFILES}),
            "rules": [VS_RULE],
            "traffic_matching_criteria": f"/Common/Shared/{identifier}-VIP_P_TCP_VS_TMC_OBJ",
        },
        "UDP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_UDP",
            "ip_protocol": "udp",
            "mask": MASK_FIXED,
            "profiles": sorted({p.format(id=identifier) for p in UDP_PROFILES}),
            "rules": [VS_RULE],
            "traffic_matching_criteria": f"/Common/Shared/{identifier}-VIP_P_UDP_VS_TMC_OBJ",
        },
    }


def build_traffic_matching_criteria_expected(identifier: str):
    return {
        "TCP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_TCP_VS_TMC_OBJ",
            "destination_address_inline": TMC_DESTINATION_ADDRESS_INLINE,
            "source_address_list": f"/Common/Shared/{identifier}_addresslist",
        },
        "UDP": {
            "name": f"/Common/Shared/{identifier}-VIP_P_UDP_VS_TMC_OBJ",
            "destination_address_inline": TMC_DESTINATION_ADDRESS_INLINE,
            "source_address_list": f"/Common/Shared/{identifier}_addresslist",
        },
    }


def excel_to_expected(excel_path: str | Path):
    excel_path = Path(excel_path)
    wb = load_workbook(excel_path, data_only=True)

    sheet_data = {}
    for sheet in SHEETS:
        if sheet not in wb.sheetnames:
            raise ValueError(f"Sheet not found: {sheet}. Available: {wb.sheetnames}")
        ws = wb[sheet]
        sheet_data[sheet] = read_sheet_values_by_identifier(ws)

    identifiers = set()
    for d in sheet_data.values():
        identifiers |= set(d.keys())

    expected = {}
    for ident in sorted(identifiers):
        expected[ident] = {

            # ここを変更
            "source_address_list": sheet_data["SourceAddress"].get(ident, []),

            "data_groups": {
                "blacklist": sheet_data["BlackList"].get(ident, []),
                "whitelist": sheet_data["WhiteList"].get(ident, []),
                "category": sheet_data["CategoryList"].get(ident, []),
            },

            "virtual_servers": build_virtual_servers_expected(ident),

            "traffic_matching_criteria": build_traffic_matching_criteria_expected(ident),
        }

    return expected


# ====== Run ======
excel_file = Path("./Subscriber.xlsx")
out_yaml = Path("./output2/output.yaml")

expected = excel_to_expected(excel_file)

out_yaml.parent.mkdir(parents=True, exist_ok=True)

with open(out_yaml, "w", encoding="utf-8") as f:
    yaml.dump(expected, f, allow_unicode=True, sort_keys=False)

print(f"Exported: {out_yaml} (identifiers={len(expected)})")


In [ ]:
#expected = excel_to_expected(excel_file)
#
## 出力ディレクトリが無ければ作成
#out_yaml.parent.mkdir(parents=True, exist_ok=True)
#
## YAML出力
#with open(out_yaml, "w", encoding="utf-8") as f:
#    yaml.dump(expected, f, allow_unicode=True, sort_keys=False)
#
#print(f"Exported: {out_yaml} (identifiers={len(expected)})")

# actual

In [ ]:
from pathlib import Path
import re
import yaml

# ====== Input / Output ======
INPUT_VS = Path("./input/tmsh-vs.txt")
INPUT_DG = Path("./input/tmsh-dg.txt")
INPUT_TMC = Path("./input/tmsh-tmc.txt")
INPUT_AL = Path("./input/tmsh-al.txt")

OUTPUT_YAML = Path("./output2/actual.yaml")


# ====== Utilities ======
def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def normalize_path(value: str) -> str:
    value = value.strip()
    if value.startswith("/Common/"):
        return value
    if value.startswith("Shared/"):
        return f"/Common/{value}"
    if value in ("tcp", "udp"):
        return f"/Common/{value}"
    return value


def sorted_unique(values):
    return sorted(set(values))


def extract_identifier_and_proto_from_vs_name(name: str):
    """
    Input:
      /Common/Shared/hoge_fuga-VIP_P_TCP
      /Common/Shared/hoge_fuga-VIP_P_UDP
    Output:
      ("hoge_fuga", "TCP") or ("hoge_fuga", "UDP")
    """
    m = re.match(r"^/Common/Shared/(.+)-VIP_P_(TCP|UDP)$", name)
    if not m:
        return None, None
    return m.group(1), m.group(2)


def extract_identifier_and_kind_from_dg_name(name: str):
    """
    Input:
      hoge_fuga_blacklist
      hoge_fuga_category
      hoge_fuga_whitelist
    Output:
      ("hoge_fuga", "blacklist") etc
    """
    m = re.match(r"^(.+)_(blacklist|category|whitelist)$", name)
    if not m:
        return None, None
    return m.group(1), m.group(2)


def extract_identifier_and_proto_from_tmc_name(name: str):
    """
    Input:
      /Common/Shared/hoge_fuga-VIP_P_TCP_VS_TMC_OBJ
      /Common/Shared/hoge_fuga-VIP_P_UDP_VS_TMC_OBJ
    Output:
      ("hoge_fuga", "TCP") or ("hoge_fuga", "UDP")
    """
    m = re.match(r"^/Common/Shared/(.+)-VIP_P_(TCP|UDP)_VS_TMC_OBJ$", name)
    if not m:
        return None, None
    return m.group(1), m.group(2)


def extract_identifier_from_address_list_name(name: str):
    """
    Input:
      /Common/Shared/hoge_fuga_addresslist
      hoge_fuga_addresslist
    Output:
      hoge_fuga
    """
    name = normalize_path(name)
    m = re.match(r"^/Common/Shared/(.+)_addresslist$", name)
    if not m:
        return None
    return m.group(1)


def ensure_identifier_root(actual: dict, identifier: str):
    if identifier not in actual:
        actual[identifier] = {
            "source_address_list": [],
            "data_groups": {
                "blacklist": [],
                "whitelist": [],
                "category": [],
            },
            "virtual_servers": {},
            "traffic_matching_criteria": {},
        }


# ====== Parsers ======
def parse_virtual_servers(text: str) -> dict:
    actual = {}

    # block extraction
    pattern = re.compile(
        r"(?ms)^ltm virtual (?P<name>\S+)\s*\{\n(?P<body>.*?)^\}",
        re.MULTILINE
    )

    for m in pattern.finditer(text):
        raw_name = m.group("name").strip()
        body = m.group("body")

        name = normalize_path(raw_name)
        identifier, proto = extract_identifier_and_proto_from_vs_name(name)
        if not identifier or not proto:
            continue

        ensure_identifier_root(actual, identifier)

        # ip-protocol
        ip_protocol_match = re.search(r"^\s*ip-protocol\s+(\S+)", body, re.MULTILINE)
        ip_protocol = ip_protocol_match.group(1).strip() if ip_protocol_match else None

        # profiles block
        profiles = []
        profiles_block = re.search(r"(?ms)^\s*profiles\s*\{\n(.*?)^\s*\}", body, re.MULTILINE)
        if profiles_block:
            for line in profiles_block.group(1).splitlines():
                line = line.strip()
                if not line:
                    continue
                # e.g. Shared/hoge_fuga_profile { }
                obj_match = re.match(r"^(\S+)\s+\{\s*\}$", line)
                if obj_match:
                    profiles.append(normalize_path(obj_match.group(1)))

        # rules block
        rules = []
        rules_block = re.search(r"(?ms)^\s*rules\s*\{\n(.*?)^\s*\}", body, re.MULTILINE)
        if rules_block:
            for line in rules_block.group(1).splitlines():
                line = line.strip()
                if not line:
                    continue
                rules.append(normalize_path(line))

        # traffic-matching-criteria
        tmc_match = re.search(r"^\s*traffic-matching-criteria\s+(\S+)", body, re.MULTILINE)
        tmc_name = normalize_path(tmc_match.group(1)) if tmc_match else None

        actual[identifier]["virtual_servers"][proto] = {
            "name": name,
            "ip_protocol": ip_protocol,
            "profiles": sorted_unique(profiles),
            "rules": sorted_unique(rules),
            "traffic_matching_criteria": tmc_name,
        }

    return actual


def parse_data_groups(text: str) -> dict:
    actual = {}

    pattern = re.compile(
        r"(?ms)^ltm data-group internal (?P<name>\S+)\s*\{\n(?P<body>.*?)^\}",
        re.MULTILINE
    )

    for m in pattern.finditer(text):
        dg_name = m.group("name").strip()
        body = m.group("body")

        identifier, kind = extract_identifier_and_kind_from_dg_name(dg_name)
        if not identifier or not kind:
            continue

        ensure_identifier_root(actual, identifier)

        records = []
        records_block = re.search(r"(?ms)^\s*records\s*\{\n(.*?)^\s*\}", body, re.MULTILINE)
        if records_block:
            for line in records_block.group(1).splitlines():
                line = line.strip()
                if not line:
                    continue
                # e.g. "aaa.com" {}
                rec_match = re.match(r'^"(.+?)"\s+\{\s*\}$', line)
                if rec_match:
                    records.append(rec_match.group(1))

        actual[identifier]["data_groups"][kind] = sorted_unique(records)

    return actual


def parse_tmc(text: str) -> dict:
    actual = {}

    pattern = re.compile(
        r"(?ms)^ltm traffic-matching-criteria (?P<name>\S+)\s*\{\n(?P<body>.*?)^\}",
        re.MULTILINE
    )

    for m in pattern.finditer(text):
        raw_name = m.group("name").strip()
        body = m.group("body")

        name = normalize_path(raw_name)
        identifier, proto = extract_identifier_and_proto_from_tmc_name(name)
        if not identifier or not proto:
            continue

        ensure_identifier_root(actual, identifier)

        dest_match = re.search(r"^\s*destination-address-inline\s+(\S+)", body, re.MULTILINE)
        source_list_match = re.search(r"^\s*source-address-list\s+(\S+)", body, re.MULTILINE)

        destination_address_inline = dest_match.group(1).strip() if dest_match else None
        source_address_list = normalize_path(source_list_match.group(1)) if source_list_match else None

        actual[identifier]["traffic_matching_criteria"][proto] = {
            "name": name,
            "destination_address_inline": destination_address_inline,
            "source_address_list": source_address_list,
        }

    return actual


def parse_address_lists(text: str) -> dict:
    actual = {}

    pattern = re.compile(
        r"(?ms)^net address-list (?P<name>\S+)\s*\{\n(?P<body>.*?)^\}",
        re.MULTILINE
    )

    for m in pattern.finditer(text):
        raw_name = m.group("name").strip()
        body = m.group("body")

        name = normalize_path(raw_name)
        identifier = extract_identifier_from_address_list_name(name)
        if not identifier:
            continue

        ensure_identifier_root(actual, identifier)

        records = []
        addresses_block = re.search(r"(?ms)^\s*address(?:es)?\s*\{\n(.*?)^\s*\}", body, re.MULTILINE)
        if addresses_block:
            for line in addresses_block.group(1).splitlines():
                line = line.strip()
                if not line:
                    continue
                # e.g. 192.168.22.0/27 { }
                rec_match = re.match(r"^(\S+)\s+\{\s*\}$", line)
                if rec_match:
                    records.append(rec_match.group(1))

        actual[identifier]["source_address_list"] = sorted_unique(records)

    return actual


# ====== Merge ======
def deep_merge_actual(base: dict, extra: dict) -> dict:
    for identifier, data in extra.items():
        ensure_identifier_root(base, identifier)

        if data.get("source_address_list"):
            base[identifier]["source_address_list"] = data["source_address_list"]

        if "data_groups" in data:
            for kind, values in data["data_groups"].items():
                base[identifier]["data_groups"][kind] = values

        if "virtual_servers" in data:
            for proto, vs_data in data["virtual_servers"].items():
                base[identifier]["virtual_servers"][proto] = vs_data

        if "traffic_matching_criteria" in data:
            for proto, tmc_data in data["traffic_matching_criteria"].items():
                base[identifier]["traffic_matching_criteria"][proto] = tmc_data

    return base


# ====== Main ======
def build_actual_from_tmsh_files(
    vs_file: Path,
    dg_file: Path,
    tmc_file: Path,
    al_file: Path,
) -> dict:
    actual = {}

    actual = deep_merge_actual(actual, parse_virtual_servers(read_text(vs_file)))
    actual = deep_merge_actual(actual, parse_data_groups(read_text(dg_file)))
    actual = deep_merge_actual(actual, parse_tmc(read_text(tmc_file)))
    actual = deep_merge_actual(actual, parse_address_lists(read_text(al_file)))

    return dict(sorted(actual.items()))


actual = build_actual_from_tmsh_files(
    vs_file=INPUT_VS,
    dg_file=INPUT_DG,
    tmc_file=INPUT_TMC,
    al_file=INPUT_AL,
)

ensure_parent(OUTPUT_YAML)
with open(OUTPUT_YAML, "w", encoding="utf-8") as f:
    yaml.dump(actual, f, allow_unicode=True, sort_keys=False)

print(f"Exported: {OUTPUT_YAML} (identifiers={len(actual)})")